In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue, detrend_ue_w_quality
from pathlib import Path
import pathlib
import argparse

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Accuracy':['gsm8k'],
    'AlignScoreInputOutput':['xsum']
}

METRICS = {
    'Accuracy': 'Accuracy',
    'AlignScoreInputOutput': 'Align Score',
    # 'Comet-wmt22-comet-da': 'Comet WMT22',
    # 'bleu_proper': 'BLEU',
        'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',

}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        "\\centering\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lc}\n"
                "\\toprule\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()


def main(type_):
    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        caption = f"Detailed PRR scores (raw and detrended) for all methods. Metric: {metric_name}."
        header = get_header(caption)
        header += "&" + "&".join([colname(dataset).replace('_', '\\_') for dataset in datasets]) + "\\\\\n"
        latex = header

        for model_type in ['base']:
            for model, model_name in MODELS.items():
                model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                latex += "\midrule\n"
                latex += "& \\multicolumn{8}{c}{" + model_title + "}\\\\\n"
                latex += "\midrule\n"
                ue_scores_100, _ , _, _,_= detrend_ue_w_quality(datasets, model, model_type, [metric], ue_methods, methods_dict, return_unprocessed=True, quality_fit_sample_size=100)
                ue_scores, _ , _, _,_= detrend_ue_w_quality(datasets, model, model_type, [metric], ue_methods, methods_dict, return_unprocessed=True, quality_fit_sample_size=500)
                ue_scores_full_train, _ , _, _,_= detrend_ue_w_quality(datasets, model, model_type, [metric], ue_methods, methods_dict, return_unprocessed=True)

                # Find best and second-best scores per dataset (column-wise)
                best_map = {}
                second_map = {}
                for col_idx in range(len(datasets)):
                    scores = []
                    for method_short in methods_dict.values():
                        raw = ue_scores[f"{method_short}_raw"][col_idx]
                        detr_100 = ue_scores_100[f"{method_short}_detr"][col_idx]
                        detr = ue_scores[f"{method_short}_detr"][col_idx]
                        detr_full = ue_scores_full_train[f"{method_short}_detr"][col_idx]
                        scores.extend([raw,detr_100, detr, detr_full])
                    sorted_scores = sorted(scores, reverse=True)
                    best_map[col_idx] = sorted_scores[0]
                    second_map[col_idx] = sorted_scores[1]

                for method_full, method_short in methods_dict.items():
                    raw_scores = ue_scores[f"{method_short}_raw"]
                    detr_scores = ue_scores[f"{method_short}_detr"]
                    detr_full_scores = ue_scores_full_train[f"{method_short}_detr"]
                    raw_row = [f"Raw {method_short}"]
                    detr_row_100 = [f"Detr {method_short} (100 samples)"]
                    detr_row = [f"Detr {method_short} (500 samples)"]
                    detr_full_row = [f"Detr {method_short} (full train)"]
                    for col_idx, (raw, detr, detr_full) in enumerate(zip(raw_scores, detr_scores, detr_full_scores)):
                        # Format raw
                        if np.isclose(raw, best_map[col_idx]):
                            raw_row.append(f"\\textbf{{{raw:.2f}}}")
                        elif np.isclose(raw, second_map[col_idx]):
                            raw_row.append(f"\\underline{{{raw:.2f}}}")
                        else:
                            raw_row.append(f"{raw:.2f}")

                        # Format detr
                        if np.isclose(detr, best_map[col_idx]):
                            detr_row.append(f"\\textbf{{{detr:.2f}}}")
                        elif np.isclose(detr, second_map[col_idx]):
                            detr_row.append(f"\\underline{{{detr:.2f}}}")
                        else:
                            detr_row.append(f"{detr:.2f}")
                        
                        if np.isclose(detr_full, best_map[col_idx]):
                            detr_full_row.append(f"\\textbf{{{detr_full:.2f}}}")
                        elif np.isclose(detr_full, second_map[col_idx]):
                            detr_full_row.append(f"\\underline{{{detr_full:.2f}}}")
                        else:
                            detr_full_row.append(f"{detr_full:.2f}")
                        
                        if np.isclose(detr_100, best_map[col_idx]):
                            detr_row_100.append(f"\\textbf{{{detr_100:.2f}}}")
                        elif np.isclose(detr_100, second_map[col_idx]):
                            detr_row_100.append(f"\\underline{{{detr_100:.2f}}}")
                        else:
                            detr_row_100.append(f"{detr_100:.2f}")


                    latex += " & ".join(raw_row) + " \\\\\n"
                    latex += " & ".join(detr_row_100) + " \\\\\n"
                    latex += " & ".join(detr_row) + " \\\\\n"
                    latex += " & ".join(detr_full_row) + " \\\\\n"
                    latex += "\\midrule\n"

        latex += footer()
        name = f'tables_rebuttal/{metric}_quality.tex'
        with open(name, 'w') as f:
            f.write(latex)


if __name__ == "__main__":
    type_='all'
    # main(type_)

/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
from utils import summarize_quality_fit  # or wherever you define it

def main(type_):
    if type_ == 'summary':
        for metric, metric_name in METRICS.items():
            datasets = DATASETS[metric]
            ue_methods = list(methods_dict.values())

            caption = f"Summary of regression fits for UE and quality vs. length. Metric: {metric_name}."
            latex = get_header(caption)
            latex += "Model & Dataset & Method & R$^2$ (Q) & MSE (Q) & Coef (Q) & R$^2$ (UE) & MSE (UE) & Coef (UE) \\\\\n\\midrule\n"

            for model_type in ['base']:
                for model, model_name in MODELS.items():
                    model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                    latex += "\\midrule\n"
                    latex += f"\\multicolumn{{9}}{{c}}{{{model_title}}} \\\\\n\\midrule\n"

                    summary = summarize_quality_fit(datasets, model, model_type, [metric], ue_methods, methods_dict)

                    for dataset in datasets:
                        short_dataset = colname(dataset)
                        for method in ue_methods:
                            stats = summary[dataset][method]
                            row = [
                                model_name,
                                short_dataset,
                                method,
                                f"{stats['quality_r2']:.2f}",
                                f"{stats['quality_mse']:.3f}",
                                f"{stats['quality_coef']:.2f}",
                                f"{stats['ue_r2']:.2f}",
                                f"{stats['ue_mse']:.3f}",
                                f"{stats['ue_coef']:.2f}",
                            ]
                            latex += " & ".join(row) + " \\\\\n"

            latex += footer()
            with open(f'tables_rebuttal/{metric}_fit_summary.tex', 'w') as f:
                f.write(latex)


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


ImportError: cannot import name 'summarize_quality_fit' from 'utils' (/home/maiya.goloburda/detrend/detrending_ue/utils.py)

In [3]:
from utils import summarize_quality_fit
import argparse

def main(type_):
    if type_ == 'summary':
        for metric, metric_name in METRICS.items():
            datasets = DATASETS[metric]
            ue_methods = list(methods_dict.values())

            caption = f"MSE and sample size (N) for UE vs. length regression fits. Metric: {metric_name}."
            latex = get_header(caption)
            latex += "Model & Dataset & Method & MSE (UE) & N \\\\\n\\midrule\n"

            for model_type in ['base']:
                for model, model_name in MODELS.items():
                    model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                    latex += "\\midrule\n"
                    latex += f"\\multicolumn{{5}}{{c}}{{{model_title}}} \\\\\n\\midrule\n"

                    summary = summarize_quality_fit(datasets, model, model_type, [metric], ue_methods, methods_dict)

                    for dataset in datasets:
                        short_dataset = colname(dataset)
                        for method in ue_methods:
                            stats = summary[dataset][method]
                            row = [
                                model_name,
                                short_dataset,
                                method,
                                f"{stats['ue_mse']:.4f}",
                                str(stats['n_train_points']),
                            ]
                            latex += " & ".join(row) + " \\\\\n"

            latex += footer()
            with open(f'tables_rebuttal/{metric}_fit_summary_ue_mse_n.tex', 'w') as f:
                f.write(latex)

if __name__ == "__main__":
    main('summary')


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fb310f16c50>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fb31079bc70>]


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fb3e5d2e680>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fb310754be0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fb3113a6e30>]


In [2]:
from utils import temp_scale_ue, isotonic_scale_ue, sigmoid_scale_ue, quadratic_scale_ue
import argparse

def main(type_):
    if type_ == 'prr_temp':
        ue_methods = list(methods_dict.values())

        for metric, metric_name in METRICS.items():
            datasets = DATASETS[metric]

            caption = f"PRR scores (raw and temperature-scaled) for UE methods. Metric: {metric_name}."
            latex = get_header(caption)
            latex += "Model & Dataset & Method & Raw PRR & Temp-Scaled PRR \\\\\n\\midrule\n"

            for model_type in ['base']:
                for model, model_name in MODELS.items():
                    model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                    latex += "\\midrule\n"
                    latex += f"\\multicolumn{{5}}{{c}}{{{model_title}}} \\\\\n\\midrule\n"

                    ue_scores, best_temperatures, _ = quadratic_scale_ue(
                        datasets, model, model_type,
                        [metric], ue_methods, methods_dict
                    )

                    for i, dataset in enumerate(datasets):
                        short_dataset = colname(dataset)
                        for method in ue_methods:
                            raw_score = ue_scores[f'{method}_raw'][i]
                            temp_score = ue_scores[f'{method}_quad'][i]
                            row = [
                                model_name,
                                short_dataset,
                                method,
                                raw_score,
                                temp_score
                            ]
                            latex += " & ".join(row) + " \\\\\n"

            latex += footer()
            with open(f'tables_rebuttal/{metric}_ue_prr_temp.tex', 'w') as f:
                f.write(latex)

if __name__ == "__main__":
    main('prr_temp')


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fc86a7ce080>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fcab407a500>]


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fc816198940>]


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fc86a7b7190>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fc86a7b7cd0>]
